# 🎵 MusicBrainz API Exploration — EDM Release Groups, Tags & Labels

This notebook explores the MusicBrainz live REST API to gather metadata for electronic music (EDM) sub-genre classification and curation analysis.

### Objectives
* Query total available records for target genre tags (`tag:edm`).
* Fetch sample release group payloads with pagination and custom rate-limiting.
* Normalize nested JSON structures into tabular DataFrames.
* Extract high-value categorical features: **Artists**, **Primary Types**, **Genre Tags**, and **Record Labels** (e.g., *Ultra Music*, *Bad Boy Records*, *Big Beat*).

## 1. Environment & Utility Imports

In [1]:
import requests
import pandas as pd
import time
import logging
import os
import ast
from collections import Counter

from data_ravers_utils.file_handler import save_df_pickle, read_df_pickle, save_df_as_csv
from data_ravers_utils import eda_utils
from notebooks.data_sources.api_helpers import fetch_raw_api_sample

## 2. API Configuration & Live Volume Check

In [2]:
# API Configuration Constants
HEADERS = {"User-Agent": "AGIes/0.1 (info@dataravers.space)"}
BASE_URL = "https://musicbrainz.org/ws/2/release-group/"
DEFAULT_TAG = "edm"
FETCH_LIMIT = 50
RATE_LIMIT_DELAY = 1.2
REQUEST_TIMEOUT=30 

In [3]:
def get_mb_tag_total_count(tag_name=DEFAULT_TAG):

    """
    Fetch the live total record count for a specific tag on MusicBrainz.
    """
    params={"query": f"tag:{tag_name}", "fmt": "json", "limit": 1}
    data = fetch_raw_api_sample(BASE_URL, params=params, headers=HEADERS, timeout=REQUEST_TIMEOUT)
    
    total_count = data.get("count", 0) if data else 0
    return total_count

# Query live total count
total_edm_records = get_mb_tag_total_count()

## 3. Fetch & Store Sample Batch

Pull a sample batch while respecting the 1 request/second rate limit, then save locally.

In [4]:
def fetch_raw_musicbrainz_sample(sample_limit, tag=DEFAULT_TAG, limit=FETCH_LIMIT):
    """Fetch a small sample batch for initial EDA without full collection wait times."""
    raw_sample = []
    
    for offset in range(0, sample_limit, limit):
        params = {
            "query": f"tag:{tag}",
            "fmt": "json",
            "limit": FETCH_LIMIT,
            "offset": offset
        }
        
        # Use your generic API helper for the individual request
        data = fetch_raw_api_sample(BASE_URL, params=params, headers=HEADERS, timeout=REQUEST_TIMEOUT)
            
        if data:
            records = data.get("release-groups", [])
            raw_sample.extend(records)
            print(f"Fetched sample offset {offset} ({len(raw_sample)} records)")
        # Keep rate limiting to prevent HTTP 503 blocks
        time.sleep(RATE_LIMIT_DELAY)
        
    return raw_sample

# Execute sample fetch
raw_sample_data = fetch_raw_musicbrainz_sample(sample_limit=15)
df_sample = pd.DataFrame({"raw_payload": raw_sample_data})
print(f"Collected {len(df_sample)} records for exploration.")

# 1. Ensure the 'data' directory exists locally
os.makedirs("data", exist_ok=True)
# Save raw sample binary pickle
save_df_pickle(df=df_sample, filename="musicbrainz_edm_raw_sample")
# Save flat CSV for quick manual viewing
save_df_as_csv(df=df_sample, filename="musicbrainz_edm_raw_sample")
print("Directory created and raw sample files successfully saved!")

Fetched sample offset 0 (50 records)
Collected 50 records for exploration.
Directory created and raw sample files successfully saved!


## 4. Reload Sample & Normalize Payload

In [5]:
# Load sample from disk
df_raw=read_df_pickle("musicbrainz_edm_raw_sample")
print("DataFrame Shape:", df_raw.shape)
display(df_raw)

# Flatten raw JSON into a temporary DataFrame for EDA
df_samples = pd.json_normalize(df_raw["raw_payload"])
# 2. Convert list and dictionary columns to strings so unique/nunique operations don't fail
for col in df_samples.columns:
    if df_samples[col].apply(lambda x: isinstance(x, (list, dict))).any():
        df_samples[col] = df_samples[col].astype(str)


DataFrame Shape: (50, 1)


,raw_payload
0,"{'id': '3bf604a1-5039-4ead-ac3e-1959884c9c97',..."
1,"{'id': '860f4808-bea7-486b-ab96-26f578c03137',..."
2,"{'id': 'ddfc24ec-3376-4ad0-8279-ca47d579fcee',..."
3,"{'id': '8a787daf-2a37-45a1-9ae1-53197eb79541',..."
4,"{'id': '4dc4fb78-e096-3f7b-9334-9defe9890ef8',..."
5,"{'id': '15425b37-1009-4a66-854e-079d96609320',..."
6,"{'id': '3b766d4f-7b6b-44a3-9b52-bff158484f40',..."
7,"{'id': '766c938e-c006-4ee0-954e-6fadf3cc78ec',..."
8,"{'id': '7ba3a993-00b1-4627-a2e2-0e56127c039a',..."
9,"{'id': '5db6d28b-7486-49a6-ac43-beed4b4162bb',..."


## 5. Exploratory Data Analysis & Tag Extraction

In [6]:
# Generate report on 500 samples
eda_utils.print_eda_report(df_samples)

# Helper function to extract tag names
def extract_tag_names(raw_tags_str):
    """Convert the stringified tags list back into just a list of tag names."""
    try:
        tags_list = ast.literal_eval(raw_tags_str)
        return [t['name'] for t in tags_list]
    except (ValueError, SyntaxError, TypeError):
        return []
# Parse tag names and compute frequencies    
df_samples['tag_names'] = df_samples['tags'].apply(extract_tag_names)
all_tags = [tag for tags_list in df_samples['tag_names'] for tag in tags_list]
tag_counts = Counter(all_tags)
display(pd.Series(tag_counts).sort_values(ascending=False).head(20))


================= Dataset =================
Dataset has shape (50, 14)

Dataset has numerical data in columns: ['score', 'count']
- Column "count" has 4 unique values.
  -- Unique values are:
 [1 2 3 4]
- Column "score" has 1 unique values.
  -- Unique values are:
 [100]

Dataset has categorical data in columns: ['id', 'type-id', 'primary-type-id', 'artist-credit-id', 'title', 'first-release-date', 'primary-type', 'secondary-types', 'secondary-type-ids', 'artist-credit', 'releases', 'tags']
- Column "id" has 50 unique values.
- Column "type-id" has 6 unique values.
  -- Unique values are:
 <StringArray>
['d6038452-8ee0-3f68-affc-2de9a1ede0b9',
 'f529b476-6e62-324f-b0aa-1f3e33d313fc',
 'dd2a21e1-0c00-3729-a7a0-de60b84eb5d1',
 '6d0c5bf6-7a33-3420-a519-44fc63eedebf',
 '0c60f497-ff81-3818-befd-abfc84a4858b',
 '6fd474e2-6b58-3102-9d17-d6f7eb7da0a0']
Length: 6, dtype: str
- Column "primary-type-id" has 3 unique values.
  -- Unique values are:
 <StringArray>
['d6038452-8ee0-3f68-affc-2de9a1ed

edm    50
dtype: int64

## 6. Extract Label Metadata for EDM Classification

Record labels serve as strong categorical predictors for EDM sub-genre classification (e.g., Anjunabeats for Trance, Drumcode for Techno). Here we look up label relations associated with our sample release groups via the MusicBrainz API.

In [10]:
def extract_release_labels(release_group_id, retries=2):
    """Fetch label names linked to a release group with basic retry logic."""
    url = "https://musicbrainz.org/ws/2/release/"
    params = {
        "release-group": release_group_id,
        "inc": "labels",
        "fmt": "json"
    }
    
    data = None
    for attempt in range(retries):
        # Increase timeout to 30 seconds to handle slow API responses
        data = fetch_raw_api_sample(url, params=params, headers=HEADERS, timeout=30)
        if data:
            break
        time.sleep(2)  # Wait before retrying on timeout/error
        
    if not data:
        return []

    labels = []
    for release in data.get("releases", []):
        for label_info in release.get("label-info", []):
            if label_info.get("label"):
                labels.append(label_info["label"]["name"])
                
    return list(set(labels))

# Test label extraction on the first 5 sample IDs
sample_ids = df_samples["id"].head(5)
label_results = {}

for rg_id in sample_ids:
    label_results[rg_id] = extract_release_labels(rg_id)
    time.sleep(RATE_LIMIT_DELAY)  # Maintain 1.2s delay between requests

# Format as DataFrame and render
df_labels = pd.DataFrame(list(label_results.items()), columns=["release_group_id", "labels"])
display(df_labels)

ERROR:root:API Request Failed for URL https://musicbrainz.org/ws/2/release/: 503 Server Error: Service Temporarily Unavailable for url: https://musicbrainz.org/ws/2/release/?release-group=860f4808-bea7-486b-ab96-26f578c03137&inc=labels&fmt=json
ERROR:root:API Request Failed for URL https://musicbrainz.org/ws/2/release/: 503 Server Error: Service Temporarily Unavailable for url: https://musicbrainz.org/ws/2/release/?release-group=ddfc24ec-3376-4ad0-8279-ca47d579fcee&inc=labels&fmt=json


,release_group_id,labels
0,3bf604a1-5039-4ead-ac3e-1959884c9c97,[Big Beat]
1,860f4808-bea7-486b-ab96-26f578c03137,[Bad Boy Records]
2,ddfc24ec-3376-4ad0-8279-ca47d579fcee,[Wolf Project]
3,8a787daf-2a37-45a1-9ae1-53197eb79541,[Ultra Music]
4,4dc4fb78-e096-3f7b-9334-9defe9890ef8,"[Bonnier Music, avex trax]"


In [ ]:
## 7. Appendix: Bulk Data Collection Function

The function below is kept for future reference when pulling the complete collection (5,746+ records). It includes explicit backoff logic for 503 Service Unavailable errors.

In [ ]:

def fetch_raw_musicbrainz_records(total_count, tag=DEFAULT_TAG):
""""
Paginate and store complete unparsed records with backoff retry handling for 503 errors."""

    raw_records = []
    for offset in range(0, total_count, FETCH_LIMIT):
        params = {"query": f"tag:{tag}", "fmt": "json", "limit": FETCH_LIMIT, "offset": offset}
        success = False
        retries = 0
        
        while not success and retries < 3:
            try:
                res = requests.get(BASE_URL, params=params, headers=HEADERS, timeout=REQUEST_TIMEOUT)
                res.raise_for_status()
                raw_records.extend(res.json().get("release-groups", []))
                success = True
            except requests.exceptions.HTTPError as e:
                if res.status_code == 503:
                    retries += 1
                    print(f"HTTP 503 hit at offset {offset}. Backing off for {retries * 5} seconds...")
                    time.sleep(retries * 5)
                else:
                    print(f"HTTP Error at offset {offset}: {e}")
                    break
            except Exception as e:
                print(f"Request failed at offset {offset}: {e}")
                break
                
        time.sleep(RATE_LIMIT_DELAY)
    return raw_records

# Execute full bulk download when ready
# raw_full_data = fetch_raw_musicbrainz_records(total_count=total_edm_records)
